# Sylvia kernel smoke test

Run all cells after changing the kernel host (`src/lang/interactive/Sylvia.Interactive.Jupyter`) or the `ext/interactive` fork. Each cell says what it expects.

Before rebuilding the kernel, shut this notebook's kernel down: a running kernel locks the DLLs in `bin\Debug`.

In [ ]:
// 1. Which kernel build is running? Expect the path under src\lang\interactive\...\bin\Debug and a recent build time.
let kernelDll = System.Reflection.Assembly.GetEntryAssembly().Location
let fsharpKernelDll = typeof<Microsoft.DotNet.Interactive.FSharp.FSharpKernel>.Assembly.Location
[ for f in [ kernelDll; fsharpKernelDll ] -> f, System.IO.File.GetLastWriteTime f ]

In [ ]:
// 2. Session-wide F# options. Expect NO FS3391 warning: there is no #nowarn in this cell or the session.
type ImplicitA() =
    static member op_Implicit(x: int) : ImplicitA = ImplicitA()
let implicits : ImplicitA list = [ 7 ]
implicits.Length

In [ ]:
// 3. #load of a script with #r "nuget: ..." references and the Sylvia DLLs. Expect no errors.
#load "../examples/proofs/Include.fsx"

In [ ]:
open Sylvia
open PropCalculus
open PredCalculus

let p, q, r = boolvar3 "p" "q" "r"

In [ ]:
// 4. A kernel-checked proof in the bare-rule style (`normalize` in the list converts Rule -> RuleApplication).
// Expect a complete proof and no FS3391 warning.
let smokeTheorem =
    ident prop_calculus ((p + (r ==> q)) == (r ==> (p + q))) [
        ident_implies_not_or r q |> at [left_branch; right_branch]
        ident_implies_not_or r (p + q) |> at_right
        normalize
    ]
smokeTheorem

In [ ]:
// 5. A deliberate warning (FS0020) to show that warnings other than FS3391 still come through. Expect ONE warning.
let ignoredResult () =
    1 + 1
    ()
ignoredResult ()